In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import pathlib
import torch
import pandas

from sklearn.metrics import f1_score


In [2]:
cell_map = {2: 'alpha',
 3: 'beta',
 5: 'ductal',
 0: 'acinar',
 4: 'delta',
 8: 'gamma',
 1: 'activated',
 6: 'endothelial',
 11: 'quiescent',
 9: 'macrophage',
 10: 'mast',
 7: 'epsilon',
 12: 'schwann',
 13: 'T-cell'}

In [39]:
def normalize_batch_name(batch_name: str) -> str:
    return "inDrop" if str(batch_name).startswith("inDrop") else str(batch_name)

In [40]:
torch.serialization.safe_globals([pandas.core.series.Series])
embedding = torch.load('../data/steering//Jun04-12-27embedding.pt')
embedding_steered = torch.load('../data/steering/Jun04-12-27embedding_steered.pt')
cell_type = torch.load('../data/steering/pancreas_celltype', weights_only=False).map(cell_map)
batch = torch.load('../data/steering/pancreas_batch', weights_only=False).map(normalize_batch_name)

In [41]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import numpy as np

embeddings = {
    "original": embedding,
    "steered": embedding_steered
}

unique_batches = batch.unique()
results = {}

for emb_name, emb in embeddings.items():
    X = emb
    y = cell_type.values
    b = batch.values
    batch_accs = {}
    for batch_val in unique_batches:
        train_idx = np.where(b != batch_val)[0]
        test_idx = np.where(b == batch_val)[0]

        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        knn = KNeighborsClassifier(n_neighbors=5)
        knn.fit(X_train, y_train)
        y_pred = knn.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average="weighted", labels=np.unique(y_test))
        batch_accs[batch_val] = {
            "accuracy": acc,
            "macro_f1": macro_f1
        }

    results[emb_name] = batch_accs

# Print results
for emb_name, batch_accs in results.items():
    print(f"Embedding: {emb_name}")
    for batch_val, metrics in batch_accs.items():
        print(
            f"  Tested on batch {batch_val}, "
            f"Test accuracy: {metrics['accuracy']:.3f}, "
            f"Weighted F1: {metrics['macro_f1']:.3f}"
        )

Embedding: original
  Tested on batch celseq, Test accuracy: 0.888, Weighted F1: 0.890
  Tested on batch celseq2, Test accuracy: 0.932, Weighted F1: 0.930
  Tested on batch fluidigmc1, Test accuracy: 0.923, Weighted F1: 0.910
  Tested on batch inDrop, Test accuracy: 0.899, Weighted F1: 0.884
  Tested on batch smarter, Test accuracy: 0.954, Weighted F1: 0.953
  Tested on batch smartseq2, Test accuracy: 0.911, Weighted F1: 0.909
Embedding: steered
  Tested on batch celseq, Test accuracy: 0.926, Weighted F1: 0.928
  Tested on batch celseq2, Test accuracy: 0.938, Weighted F1: 0.936
  Tested on batch fluidigmc1, Test accuracy: 0.920, Weighted F1: 0.922
  Tested on batch inDrop, Test accuracy: 0.918, Weighted F1: 0.912
  Tested on batch smarter, Test accuracy: 0.971, Weighted F1: 0.972
  Tested on batch smartseq2, Test accuracy: 0.911, Weighted F1: 0.909


In [42]:
# Build a transposed LaTeX table comparing scGPT fine-tuned vs scGPT steered.
metric_key = "macro_f1"  # Change to "accuracy" if you want the accuracy table.
metric_label = "Weighted F1"



fine_tuned_series = pd.Series(
    {batch_name: metrics[metric_key] for batch_name, metrics in results["original"].items()}
)
steered_series = pd.Series(
    {batch_name: metrics[metric_key] for batch_name, metrics in results["steered"].items()}
)

metric_df = pd.DataFrame(
    {
        "batch": fine_tuned_series.index,
        "scGPT fine-tuned": fine_tuned_series.values,
        "scGPT steered": steered_series.reindex(fine_tuned_series.index).values,
    }
)

# Merge inDrop1-4 into a single inDrop row by averaging across them.
metric_df["batch"] = metric_df["batch"].map(normalize_batch_name)
metric_df = metric_df.groupby("batch", as_index=True).mean(numeric_only=True).sort_index()

metric_df["$\\Delta$ (steered - fine-tuned)"] = (
    metric_df["scGPT steered"] - metric_df["scGPT fine-tuned"]
)

mean_row = pd.DataFrame(metric_df.mean(numeric_only=True)).T
mean_row.index = ["Mean"]
latex_df = pd.concat([metric_df, mean_row]).T

latex_table = latex_df.to_latex(
    index=True,
    float_format="%.3f",
    caption=(
        f"Transposed cross-batch KNN {metric_label} comparison between scGPT "
        "fine-tuned and scGPT steered embeddings (inDrop1-4 combined)."
    ),
    label="tab:scgpt_finetuned_vs_steered_transposed",
)

print(latex_table)

\begin{table}
\caption{Transposed cross-batch KNN Weighted F1 comparison between scGPT fine-tuned and scGPT steered embeddings (inDrop1-4 combined).}
\label{tab:scgpt_finetuned_vs_steered_transposed}
\begin{tabular}{lrrrrrrr}
\toprule
 & celseq & celseq2 & fluidigmc1 & inDrop & smarter & smartseq2 & Mean \\
\midrule
scGPT fine-tuned & 0.890 & 0.930 & 0.910 & 0.884 & 0.953 & 0.909 & 0.913 \\
scGPT steered & 0.928 & 0.936 & 0.922 & 0.912 & 0.972 & 0.909 & 0.930 \\
$\Delta$ (steered - fine-tuned) & 0.038 & 0.006 & 0.013 & 0.028 & 0.019 & 0.000 & 0.017 \\
\bottomrule
\end{tabular}
\end{table}

